In [ ]:
import pandas as pd
import os
from pathlib import Path
import re

PROJECT_ROOT = Path("/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans")
target_base_dir = str(PROJECT_ROOT / "data" / "implicit_connective_data" / "disrpt_inputs")


def parse_conllu_file(filepath):
    # Parse CoNLL-U file; sent_id format is segment_id.sentence_id (e.g. 0.0, 0.1, 1.0)
    documents = {}
    current_doc_id = None
    current_sent_id = None
    current_segment_id = None
    current_tokens = []
    current_text = ""

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line.startswith('# newdoc id ='):
                current_doc_id = line.split('=')[1].strip()
                if current_doc_id not in documents:
                    documents[current_doc_id] = {'segments': {}}

            elif line.startswith('# text ='):
                current_text = line.split('=', 1)[1].strip()

            elif line.startswith('# sent_id ='):
                sent_id_str = line.split('=')[1].strip()
                if '.' in sent_id_str:
                    parts = sent_id_str.split('.')
                    current_segment_id = int(parts[0])
                    current_sent_id = int(parts[1])
                else:
                    current_segment_id = int(sent_id_str)
                    current_sent_id = 0
                current_tokens = []

            elif line and not line.startswith('#'):
                parts = line.split('\t')
                if len(parts) >= 10:
                    token_id = parts[0]
                    token_text = parts[1]
                    if '.' not in token_id and '-' not in token_id:
                        current_tokens.append((token_id, token_text))

            elif line == "" and current_doc_id and current_segment_id is not None:
                if current_tokens:
                    if current_segment_id not in documents[current_doc_id]['segments']:
                        documents[current_doc_id]['segments'][current_segment_id] = {
                            'sentences': [], 'tokens': []
                        }
                    documents[current_doc_id]['segments'][current_segment_id]['sentences'].append({
                        'sent_id': current_sent_id,
                        'text': current_text,
                        'tokens': current_tokens.copy()
                    })
                current_tokens = []
                current_text = ""
                current_sent_id = None

    return documents


def calculate_segment_spans(documents):
    doc_spans = {}
    for doc_id, doc_data in documents.items():
        spans = []
        token_counter = 1
        for seg_id in sorted(doc_data['segments'].keys()):
            segment = doc_data['segments'][seg_id]
            segment_texts = []
            segment_token_count = 0
            for sentence in segment['sentences']:
                segment_texts.append(sentence['text'])
                segment_token_count += len(sentence['tokens'])

            if segment_token_count > 0:
                spans.append({
                    'segment_id': seg_id,
                    'span': f"{token_counter}-{token_counter + segment_token_count - 1}",
                    'text': ' '.join(segment_texts),
                    'start': token_counter,
                    'end': token_counter + segment_token_count - 1
                })
                token_counter += segment_token_count
        doc_spans[doc_id] = spans
    return doc_spans


def generate_rels_content(doc_spans):
    # Pairs consecutive segments and writes DISRPT .rels format
    header = ["doc", "unit1_toks", "unit2_toks", "unit1_txt", "unit2_txt",
              "u1_raw", "u2_raw", "s1_toks", "s2_toks", "unit1_sent", "unit2_sent",
              "dir", "rel_type", "orig_label", "label"]
    rels_lines = ["\t".join(header)]

    for doc_id, spans in doc_spans.items():
        for i in range(len(spans) - 1):
            seg1, seg2 = spans[i], spans[i + 1]
            row = [
                doc_id,
                seg1['span'], seg2['span'],
                seg1['text'], seg2['text'],
                seg1['text'], seg2['text'],
                seg1['span'], seg2['span'],
                seg1['text'], seg2['text'],
                "1<2", "UNKNOWN", "UNKNOWN", "UNKNOWN"
            ]
            rels_lines.append("\t".join(row))

    return "\n".join(rels_lines)


def process_conllu_files():
    base_path = Path(target_base_dir)

    conllu_files = []
    for folder in base_path.iterdir():
        if folder.is_dir():
            conllu_path = folder / "eng.pdtb.gum" / "eng.pdtb.gum_test.conllu"
            if conllu_path.exists():
                conllu_files.append((folder.name, conllu_path))

    print(f"Found {len(conllu_files)} CoNLL-U files")

    for folder_name, conllu_file in sorted(conllu_files):
        print(f"Processing {folder_name}...")

        try:
            documents = parse_conllu_file(conllu_file)
            print(f"  {len(documents)} documents")

            if not documents:
                print(f"  No documents found, skipping")
                continue

            doc_spans = calculate_segment_spans(documents)
            total_pairs = sum(max(0, len(spans) - 1) for spans in doc_spans.values())
            print(f"  {total_pairs} segment pairs")

            if total_pairs == 0:
                rels_content = "\t".join(["doc", "unit1_toks", "unit2_toks", "unit1_txt", "unit2_txt",
                                          "u1_raw", "u2_raw", "s1_toks", "s2_toks", "unit1_sent", "unit2_sent",
                                          "dir", "rel_type", "orig_label", "label"])
            else:
                rels_content = generate_rels_content(doc_spans)

            rels_file = conllu_file.parent / "eng.pdtb.gum_test.rels"
            with open(rels_file, 'w', encoding='utf-8') as f:
                f.write(rels_content)
            print(f"  Created: {rels_file}")

        except Exception as e:
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()

    print("Done.")

In [6]:
process_conllu_files()

Found 14 CoNLL-U files:

Processing claude45_large...
  File: /home/xilini/disrpt25-task-OLD/data_v2/claude45_large/eng.pdtb.gum/eng.pdtb.gum_test.conllu
  Found 60 documents
  Generated 285 segment pairs
  First doc 'story_10486' has 6 segments
  ✓ Created: /home/xilini/disrpt25-task-OLD/data_v2/claude45_large/eng.pdtb.gum/eng.pdtb.gum_test.rels
  Sample (first 2 pairs):
    HEADER: doc	unit1_toks	unit2_toks	unit1_txt	unit2_txt	u1_raw	u2_raw	s1_toks	s2_toks	unit...
    PAIR 1: story_10486 | seg 1-52 -> 53-100
             txt1: A masked figure dressed entirely in black tactical gear stan...
             txt2: The armed intruder bursts into a modern corporate office whe...
    PAIR 2: story_10486 | seg 53-100 -> 101-151
             txt1: The armed intruder bursts into a modern corporate office whe...
             txt2: Robert clutches his head in anguish, his hands pressed again...

Processing claude45_original...
  File: /home/xilini/disrpt25-task-OLD/data_v2/claude45_original/eng.pd